# 第5章　学習の仕組み ― 損失関数・最適化・評価

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 損失関数をコードで書く

In [ ]:
import torch.nn as nn
import torch, torch.nn as nn

# クラスごとの重み（少ないクラスほど大きく。例：背景1.0 / 病変5.0）
weights = torch.tensor([1.0, 5.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
def dice_loss(pred, target, eps=1e-6):
    pred = pred.sigmoid()                       # 0〜1の確率へ
    inter = (pred * target).sum()               # 重なり
    union = pred.sum() + target.sum()           # 予測+正解の面積
    return 1 - (2 * inter + eps) / (union + eps) # 1 - Dice係数

In [ ]:
from monai.losses import DiceCELoss, TverskyLoss
criterion = DiceCELoss(to_onehot_y=True, softmax=True)   # Dice + 交差エントロピー
# criterion = TverskyLoss(to_onehot_y=True, softmax=True, alpha=0.3, beta=0.7)  # 偽陰性(見逃し)を重く罰する

## 数字で追う ― ソフトマックスから交差エントロピー、そしてFocal損失へ

In [ ]:
import torch.nn.functional as F
import torch, torch.nn.functional as F

z = torch.tensor([[2.0, 1.0, 0.1]])
print(F.softmax(z, dim=1))                       # tensor([[0.6590, 0.2424, 0.0986]])
print(F.cross_entropy(z, torch.tensor([0])))     # tensor(0.4170) ← 生ロジットを渡す

## 学習率スケジューリング ― 歩幅を旅の途中で変える

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

for epoch in range(num_epochs):
    train_one_epoch(model, train_loader, optimizer)   # 第4章の学習ループ
    scheduler.step()                                   # 学習率を1エポック分だけ更新
    print(f"epoch {epoch}: lr = {scheduler.get_last_lr()[0]:.2e}")

## 手を動かす ― 損失が下がる様子を印字し、学習率の効果を体感する

In [ ]:
import torch

def run(lr, steps=6):
    w = torch.tensor(0.0, requires_grad=True)   # でたらめな初期値からスタート
    for step in range(steps):
        loss = (w - 3.0) ** 2                    # 順伝播：ズレの2乗が損失
        loss.backward()                          # 逆伝播：勾配 dL/dw を計算
        with torch.no_grad():
            w -= lr * w.grad                     # 更新：勾配の逆向きに一歩（勾配降下）
        w.grad.zero_()                           # 次のステップのためリセット
        print(f"  step {step}: w={w.item():.3f}, loss={loss.item():.3f}")

print("学習率 0.1（ちょうど良い）");        run(0.1)
print("学習率 0.01（小さすぎ：進みが遅い）"); run(0.01)
print("学習率 1.1（大きすぎ：発散する）");    run(1.1)

## 過学習を抑える道具立て ― 正則化・ドロップアウト・データ拡張

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

## ドロップアウトを数字で追う ― 「$1/(1-p)$ 倍」の正体

In [ ]:
# 2次元（torchvision）: 学習用は拡張あり、検証用は拡張なし
import torch
from torchvision.transforms import v2
train_tf = v2.Compose([
    v2.ToImage(),                          # PIL/ndarray をテンソルへ（これがないと後段が素通りする）
    v2.RandomRotation(15),                 # ±15度の回転
    v2.RandomResizedCrop(224, scale=(0.8, 1.0)),
    v2.ColorJitter(brightness=0.1),        # 明るさを控えめに変える
    v2.ToDtype(torch.float32, scale=True),
])

# 3次元CT（MONAI）: 拡張だけを抜き出した例（向きの統一とHUクリップは前処理側で済ませておく）
from monai import transforms as T
train_tf3d = T.Compose([
    T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),  # 空間変形は画像とラベルに同時適用（軸0=左右。部位で可否を判断）
    # 90度回転は横断面の向きを変える。RAS正規化後の体幹CT/MRIでは実際には得られない向きに
    # なるので使わない。四肢・単純X線・病理パッチなど向きの自由度がある対象に限る（第25章）。
    # T.RandRotate90d(keys=["image", "label"], prob=0.3),
    T.RandGaussianNoised(keys="image", prob=0.2),
])

## 手を動かす ― データ拡張の「効き方」を1枚の画像で並べて見る

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, torch, matplotlib.pyplot as plt
from torchvision.transforms import v2

# 合成画像：灰色の背景に、右上寄りの目立たない病変(ぼんやり明るい塊)を一つ置く
yy, xx = np.mgrid[0:96, 0:96]
img = np.full((96, 96), 0.30, np.float32)
img += 0.55 * np.exp(-(((xx - 60)**2 + (yy - 38)**2) / (2 * 11**2)))
t = torch.from_numpy(img).clamp(0, 1)[None]     # (1, H, W)

torch.manual_seed(0)
augs = {                                    # 図中は英語（日本語は□に化ける）
    "raw":            lambda x: x,
    "hflip":          v2.RandomHorizontalFlip(p=1.0),
    "rotate +15deg":  v2.RandomRotation((15, 15), interpolation=v2.InterpolationMode.BILINEAR),
    "bright x1.4":    v2.ColorJitter(brightness=(1.4, 1.4)),
    "zoom crop":      v2.RandomResizedCrop(96, scale=(0.5, 0.5), antialias=True),
    "gaussian noise": lambda x: (x + 0.08 * torch.randn_like(x)).clamp(0, 1),
}

fig, axes = plt.subplots(1, len(augs), figsize=(15, 3))
for ax, (name, f) in zip(axes, augs.items()):
    ax.imshow(f(t)[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(name); ax.axis("off")
plt.tight_layout()

## 5.8　学習の全体像 ― 一つのコードで通して眺める

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader

# 1) データを準備（学習用・検証用に分ける）
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)

# 2) モデル・損失・最適化を用意
model     = build_model().to(device)          # GPUへ載せる
criterion = nn.CrossEntropyLoss()             # 分類の損失関数
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_score = float("-inf")                     # 0 で始めると、スコアが0のときや負の指標で
                                               # 一度も保存されないまま終わる
for epoch in range(50):                        # 50エポック繰り返す
    # --- 学習フェーズ ---
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                  # 勾配をリセット
        outputs = model(images)                # 順伝播（予測）
        loss = criterion(outputs, labels)      # 誤差を計算
        loss.backward()                        # 逆伝播（勾配計算）
        optimizer.step()                       # 重みを更新

    # --- 検証フェーズ ---
    model.eval()
    score = evaluate(model, val_loader)        # 感度・AUCなどを計算
    if score > best_score:                     # 最良モデルだけ保存
        best_score = score
        torch.save(model.state_dict(), "best_model.pth")

## 応用ミニ演習 ― 設計をコードに翻訳する（解答例つき）

In [ ]:
import torch
counts = torch.tensor([9500., 500.])          # [背景, 病変]
w = counts.sum() / counts                       # → [1.05, 20.0]
w = w / w.min()                                 # 見やすく正規化 → [1.0, 19.0]
criterion = torch.nn.CrossEntropyLoss(weight=w)
print(w)   # tensor([ 1., 19.])

In [ ]:
from monai.losses import TverskyLoss
criterion = TverskyLoss(to_onehot_y=True, softmax=True, alpha=0.25, beta=0.75)   # 偽陰性(見逃し)を3倍重く

In [ ]:
import math
best, patience, wait = float("-inf"), 5, 0    # 0.0 で始めると、スコア0のとき一度も保存されない
for epoch in range(100):
    train_one_epoch(...)
    score = evaluate(...)                 # 検証スコア（APなど）。非有限値は失敗扱いにする
    if math.isfinite(score) and score > best:
        best, wait = score, 0             # 更新できたらカウンタをリセットし保存
        torch.save(model.state_dict(), "best.pth")
    else:
        wait += 1
        if wait >= patience:              # 5回連続で更新なし → 打ち切り
            print(f"early stop at epoch {epoch}"); break
if not math.isfinite(best):   # 第22章と同じ約束（assert は -O で無効化されるので使わない）
    raise RuntimeError("有効な検証スコアが一度も得られていない")